# VAE-GAN Hybrid Research: Conditional Image Synthesis on CIFAR-10
### Combining VAE Latent Stability with GAN Perceptual Quality

This notebook trains and benchmarks:
1. **Baseline Conditional VAE (CVAE)**
2. **Baseline Conditional GAN (cGAN)**
3. **Proposed Unified Conditional VAE-GAN Hybrid**

Ensure you select a GPU runtime in Colab (**Runtime -> Change runtime type -> T4 GPU / A100**).

In [ ]:
# 1. Environment & GPU Verification
!nvidia-smi
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

In [ ]:
# 2. Clone or setup directory structure
!git clone https://github.com/your-username/vae-gan-hybrid.git || echo "Working in local directory"
%cd vae-gan-hybrid || %cd /content/vae-gan-hybrid || true
!pip install -r requirements.txt

In [ ]:
# 3. Run Unit Tests to Verify Tensor Shapes & Gradient Flow
!python -m unittest discover tests

In [ ]:
# 4. Train the VAE-GAN Hybrid Model
!python training/train_hybrid.py --config configs/vae_gan_cifar10.yaml

In [ ]:
# 5. Visualize Generated Samples & Latent Interpolations
import matplotlib.pyplot as plt
from PIL import Image
import glob

sample_files = sorted(glob.glob("./outputs/hybrid/*.png"))
if sample_files:
    latest_sample = sample_files[-1]
    print(f"Displaying latest sample grid: {latest_sample}")
    img = Image.open(latest_sample)
    plt.figure(figsize=(12, 6))
    plt.imshow(img)
    plt.axis("off")
    plt.title("VAE-GAN Hybrid: Real | Reconstructed | Prior Samples")
    plt.show()

In [ ]:
# 6. Benchmark Evaluation Suite (PSNR, SSIM, LPIPS)
import yaml
from data.dataloaders import get_cifar10_loaders
from models.conditional_vae_gan import ConditionalVAEGAN
from evaluation.compare_models import benchmark_reconstruction, format_benchmark_table

with open("configs/vae_gan_cifar10.yaml", "r") as f:
    cfg = yaml.safe_load(f)

_, test_loader = get_cifar10_loaders(data_dir="./data/raw", batch_size=64)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
hybrid = ConditionalVAEGAN().to(device)

# Benchmark evaluation
results = {"VAE-GAN Hybrid": benchmark_reconstruction(hybrid, test_loader, device=device)}
print(format_benchmark_table(results))